# C1 recurrent hybrid, full tuning

**Reconstructed study companion.** The code and numeric evidence come from the archived run; these cells were assembled later and carry no saved training outputs.

## Problem and data

Phase 2 extracts 12 legal entity types as exact character spans. Train folds 1 and 2 contain 70 labeled documents; 58 were used for fitting and 12 held out for local validation. The official test fold contains 35 documents with no gold labels in the package.

## Why this model?

The C1 backbone combines a 10-layer encoder with a four-layer GRU decoder and cross-attention. Phase 2 updates all model parameters, then uses the shared character BiGRU and BIO classifier.

## Architecture

![Implementation path](../figures/architecture.svg)

Legal text → Own byte BPE → 10-layer encoder → 4-layer GRU + cross-attn → Character BiGRU → 25 BIO logits → Span decoder

## Tensor route

A token sequence `ids: [B,T]` enters the pretrained backbone. Character-to-token indices `mapping: [B,C]` gather contextual token states for every source character. A 32-dimensional character embedding is concatenated with those states. The bidirectional character GRU yields `[B,C,256]`; the classifier emits 25 BIO scores (outside + beginning/inside for 12 labels). D222 and HeaderCRF variants add learned sequence transitions and boundary signals. The consensus variant operates on complete span triples instead of tensors.

## Training and selection

Original source and config are in `../code/`. The run evidence under `../evidence/` records optimizer configuration, checkpoint identity, epoch selection, and local validation. Do not treat the code cells in this companion as a re-run of the historical training job.

In [ ]:
from pathlib import Path
import json
root = Path('..')
record = json.loads((root/'evidence'/'run_summary.json').read_text())
print(record['model'], record['local_validation_macro_f1'])

## Recovered result

Held-out strict macro-F1: **0.357207**. This is a local 12-document validation number, **not** a Codabench hidden-test score. Export status: Pre-deadline export.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
p = next((Path('..')/'submission').glob('*.zip'))
with ZipFile(p) as z:
    assert z.namelist() == ['prediction.json'] and z.testzip() is None
    predictions = json.loads(z.read('prediction.json'))
print('test documents:', len(predictions))

## Reading trail

Start with `../README.md`, then `../docs/TECHNICAL_REFERENCE.md` and `../docs/CODE_MAP.md`. The archived selected weights are too large for this study ZIP; their original complete archive and SHA provenance are named in the evidence summary. Paths in original configs refer to the old training machine and must be mapped to local copies before rerunning.